### 1. Divisão entre treino e teste

In [ ]:
# 1. Divisão entre treino e teste

# 80% dos dados serão utilizados para treinamento.
# 20% dos dados serão utilizados para teste.
# O conjunto de teste simula dados que o algoritmo ainda não conhece.

# Fixa a semente para garantir que o resultado seja reproduzível.
torch.manual_seed(42)

# Seleção das características da pétala:
# coluna 2 -> comprimento da pétala
# coluna 3 -> largura da pétala
X_petala = X[:, 2:4]

# Conversão para tensores PyTorch
X_t = torch.tensor(
    X_petala,
    dtype=torch.float32,
    device=device
)

y_t = torch.tensor(
    y,
    dtype=torch.long,
    device=device
)

# Quantidade total de amostras
n = X_t.shape[0]

# Embaralhamento dos índices
perm = torch.randperm(n)

# Definição da quantidade de dados para treinamento
n_train = int(0.8 * n)

# Separação dos índices
train_idx = perm[:n_train]
test_idx = perm[n_train:]

# Criação dos conjuntos de treino e teste
X_train = X_t[train_idx]
y_train = y_t[train_idx]

X_test = X_t[test_idx]
y_test = y_t[test_idx]

print("Treino:", X_train.shape)
print("Teste:", X_test.shape)

### 2. Cálculo das distâncias

In [ ]:
# 2. Cálculo das distâncias

def calcula_distancias(X_train, X_test):
    """
    Calcula a distância euclidiana entre cada amostra
    do conjunto de teste e cada amostra do conjunto de treino.
    """
    
    return torch.cdist(X_test, X_train, p=2)


# Matriz de distâncias:
# linhas -> amostras de teste
# colunas -> amostras de treino
dists = calcula_distancias(X_train, X_test)

print("Formato da matriz de distâncias:", dists.shape)

### 3. Classificação utilizando k-NN

In [ ]:
# 3. Classificação utilizando k-NN

def knn_predict(dists, y_train, k):
    """
    Classifica as amostras de teste utilizando
    os k vizinhos mais próximos.
    """

    # Obtém os índices dos k menores valores de distância.
    _, idx_vizinhos = torch.topk(
        dists,
        k,
        largest=False,
        dim=1
    )

    # Obtém os rótulos dos vizinhos selecionados.
    rotulos_vizinhos = y_train[idx_vizinhos]

    # Realiza uma votação entre os vizinhos.
    # O rótulo que aparecer mais vezes será a previsão.
    preds = torch.mode(
        rotulos_vizinhos,
        dim=1
    ).values

    return preds


# Classificação utilizando k = 5
preds = knn_predict(
    dists,
    y_train,
    k=5
)

# Exibe as previsões das 10 primeiras amostras
print("Previsões:", preds[:10])

### 4. Cálculo da acurácia

In [ ]:
# 4. Avaliação do modelo

def acuracia(preds, y_test):
    """
    Calcula a proporção de previsões corretas.
    """
    
    return (preds == y_test).float().mean().item()


# Cálculo da acurácia utilizando k = 5
acc = acuracia(preds, y_test)

print(f"Acurácia (k=5): {acc:.4f}")

### 5. Testando diferentes valores de K

In [ ]:
# 5. Experimento com diferentes valores de k

# Valores de k que serão testados
valores_k = [1, 3, 5, 7, 9]

# Armazena os resultados
resultados = []

for k in valores_k:
    # Realiza a classificação
    preds_k = knn_predict(dists, y_train, k)

    # Calcula a acurácia
    acc_k = acuracia(preds_k, y_test)

    # Armazena o resultado
    resultados.append((k, acc_k))


# Organização dos resultados em uma tabela
df_resultados = pd.DataFrame(
    resultados,
    columns=["k", "acuracia"]
)

print(df_resultados)

### 6. Visualização das fronteiras de decisão

In [ ]:
# 6. Visualização das fronteiras de decisão

import numpy as np
import matplotlib.pyplot as plt

# Valor de k utilizado na visualização
k_escolhido = 5

# Limites do gráfico
x_min = X_petala[:, 0].min() - 0.5
x_max = X_petala[:, 0].max() + 0.5

y_min = X_petala[:, 1].min() - 0.5
y_max = X_petala[:, 1].max() + 0.5

# Criação de uma grade de pontos 2D
xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 200),
    np.linspace(y_min, y_max, 200)
)

# Conversão dos pontos da grade para Tensor
grid_points = torch.tensor(
    np.c_[xx.ravel(), yy.ravel()],
    dtype=torch.float32,
    device=device
)

# Calcula as distâncias entre os pontos da grade
# e todas as amostras do dataset.
dists_grid = calcula_distancias(
    X_t,
    grid_points
)

# Classifica cada ponto da grade
preds_grid = knn_predict(
    dists_grid,
    y_t,
    k_escolhido
)

# Reconstrói as previsões no formato da grade
Z = preds_grid.reshape(xx.shape)

# Plotagem das fronteiras de decisão
plt.figure(figsize=(8, 6))

plt.contourf(
    xx,
    yy,
    Z.cpu(),
    alpha=0.3,
    cmap="viridis"
)

scatter = plt.scatter(
    X_petala[:, 0],
    X_petala[:, 1],
    c=y,
    cmap="viridis",
    edgecolor="k",
    s=40
)

plt.xlabel("Comprimento da pétala (cm)")
plt.ylabel("Largura da pétala (cm)")
plt.title(f"Fronteiras de decisão k-NN (k={k_escolhido})")

plt.legend(
    handles=scatter.legend_elements()[0],
    labels=list(nomes)
)

plt.show()